In [23]:
import numpy as np
from matplotlib import pyplot as plt
from electronic import *
from analytical import *
from integrator import *
from trajectory import *

In [25]:
v_init = -0.002
c = 0.0033
s = 0.66

LZ_H = LandauZener(s=s, vc=c)

dt = 20
t = np.arange(0, 8269, dt)

q_init = 10.0
coeff_init = np.array([0.0, 1.0], dtype=complex)
mass = 1836.15
C_init = LZ_H.eigvecs(q_init)

max_timestep = 800

frame_init = Snapshot(positions=q_init,
                      velocities=v_init,
                      coefficients=coeff_init,
                      active_state=1,
                      gauge=C_init,
                      mass=mass)

traj = Trajectory()
traj.append(frame_init)

for step in range(max_timestep):
    snapshot_curr = traj[-1]
    q_curr = snapshot_curr.positions
    v_curr = snapshot_curr.velocities
    coeff_curr = snapshot_curr.coefficients
    mass = snapshot_curr.mass
    active_state = snapshot_curr.active_state

    v_half = verlet_v(dt=dt, model=LZ_H, mass=mass, v_curr=v_curr, q_curr=q_curr,
                      active_state=active_state)
    q_next = verlet_X(dt=dt, v_half=v_half, q_curr=q_curr)
    v_next = verlet_v(dt=dt, model=LZ_H, mass=mass, v_curr=v_half, q_curr=q_next
                      , active_state=active_state)
    
    C_next, coeff_next = local_diabatisation(model=LZ_H,
                                             snapshot=snapshot_curr,
                                             q_next=q_next,
                                             dt=dt)
    Sz_curr = sz_from_coeff(coeff_curr)
    Sz_next = sz_from_coeff(coeff_next)

    if Sz_curr * Sz_next < 0:
        tau_M = hop_search(dt=dt,
                           snapshot=snapshot_curr,
                           model=LZ_H)
        v_half_1 = verlet_v(dt=tau_M, model=LZ_H, mass=mass, v_curr=v_curr, q_curr=q_curr,
                      active_state=active_state)
        q_next_1 = verlet_X(dt=tau_M, q_curr=q_curr, v_half=v_half_1)
        v_next_1 = verlet_v(dt=tau_M, model=LZ_H, mass=mass, v_curr=v_half_1, q_curr=q_next_1,
                      active_state=active_state)
        C_next_1, coeff_next_1 = local_diabatisation(model=LZ_H,
                                                     snapshot=snapshot_curr,
                                                     q_next=q_next_1,
                                                     dt=tau_M)
        snapshot_1 = Snapshot(positions=q_next_1,
                              velocities=v_next_1,
                              coefficients=coeff_next_1,
                              active_state=active_state,
                              mass=mass,
                              gauge=C_next_1)
        traj.append(snapshot_1)

        v_new, is_hop = velocity_rescaling(model=LZ_H, snapshot=snapshot_1)
        if is_hop:
            active_state_new = 1 - active_state
        else:
            active_state_new = active_state

        dt_R = dt - tau_M
        v_half_2 = verlet_v(dt=dt_R, model=LZ_H, mass=mass,
                            v_curr=v_new,
                            q_curr=q_next_1,
                            active_state=active_state_new)
        q_next_2 = verlet_X(dt=dt_R,
                            q_curr=q_next_1,
                            v_half=v_half_2)
        v_next_2 = verlet_v(dt=dt_R, model=LZ_H,
                            mass=mass,
                            v_curr=v_half_2,
                            q_curr=q_next_2,
                            active_state=active_state_new)
        C_next_2, coeff_next_2 = local_diabatisation(model=LZ_H,
                                                     snapshot=snapshot_1,
                                                     q_next=q_next_2,
                                                     dt=dt_R)
        
        snapshot_2 = Snapshot(positions=q_next_2,
                              velocities=v_next_2,
                              coefficients=coeff_next_2,
                              active_state=active_state_new,
                              gauge=C_next_2,
                              mass=mass)
        traj.append(snapshot_2)
    else:
        snapshot_next = Snapshot(positions=q_next,
                                 velocities=v_next,
                                 coefficients=coeff_next,
                                 active_state=active_state,
                                 gauge=C_next,
                                 mass=mass)
        traj.append(snapshot_next)

print('last position', traj[-1].positions)
print('last velocity', traj[-1].velocities)
print('last active state', traj[-1].active_state)
print('last coefficients', traj[-1].coefficients)
print('last gauge', traj[-1].gauge)

Hop search finished after 31 iterations
last position -46003.59707967745
last velocity -5.751406570334704
last active state 0
last coefficients [-0.88941136-0.45544895j  0.00774126-0.03812806j]
last gauge [[-1.08687153e-07  1.00000000e+00]
 [ 1.00000000e+00  1.08687153e-07]]


In [26]:
v_init_rev = 5.751406570334704
c = 0.0033
s = 0.66

LZ_H = LandauZener(s=s, vc=c)

dt = 20
t = np.arange(0, 8269, dt)

q_init_rev = -46003.59707967745
coeff_init_rev = np.array([-0.88941136+0.45544895j,  0.00774126+0.03812806j], dtype=complex)
mass = 1836.15
C_init_rev = traj[-1].gauge

max_timestep = 800

frame_init_rev = Snapshot(positions=q_init_rev,
                      velocities=v_init_rev,
                      coefficients=coeff_init_rev,
                      active_state=0,
                      gauge=C_init_rev,
                      mass=mass)

traj_rev = Trajectory()
traj_rev.append(frame_init_rev)

for step in range(max_timestep):
    snapshot_curr = traj_rev[-1]
    q_curr = snapshot_curr.positions
    v_curr = snapshot_curr.velocities
    coeff_curr = snapshot_curr.coefficients
    mass = snapshot_curr.mass
    active_state = snapshot_curr.active_state

    v_half = verlet_v(dt=dt, model=LZ_H, mass=mass, v_curr=v_curr, q_curr=q_curr,
                      active_state=active_state)
    q_next = verlet_X(dt=dt, v_half=v_half, q_curr=q_curr)
    v_next = verlet_v(dt=dt, model=LZ_H, mass=mass, v_curr=v_half, q_curr=q_next
                      , active_state=active_state)
    
    C_next, coeff_next = local_diabatisation(model=LZ_H,
                                             snapshot=snapshot_curr,
                                             q_next=q_next,
                                             dt=dt)
    Sz_curr = sz_from_coeff(coeff_curr)
    Sz_next = sz_from_coeff(coeff_next)

    if Sz_curr * Sz_next < 0:
        tau_M = hop_search(dt=dt,
                           snapshot=snapshot_curr,
                           model=LZ_H)
        v_half_1 = verlet_v(dt=tau_M, model=LZ_H, mass=mass, v_curr=v_curr, q_curr=q_curr,
                      active_state=active_state)
        q_next_1 = verlet_X(dt=tau_M, q_curr=q_curr, v_half=v_half_1)
        v_next_1 = verlet_v(dt=tau_M, model=LZ_H, mass=mass, v_curr=v_half_1, q_curr=q_next_1,
                      active_state=active_state)
        C_next_1, coeff_next_1 = local_diabatisation(model=LZ_H,
                                                     snapshot=snapshot_curr,
                                                     q_next=q_next_1,
                                                     dt=tau_M)
        snapshot_1 = Snapshot(positions=q_next_1,
                              velocities=v_next_1,
                              coefficients=coeff_next_1,
                              active_state=active_state,
                              mass=mass,
                              gauge=C_next_1)
        traj_rev.append(snapshot_1)

        v_new, is_hop = velocity_rescaling(model=LZ_H, snapshot=snapshot_1)
        if is_hop:
            active_state_new = 1 - active_state
        else:
            active_state_new = active_state

        dt_R = dt - tau_M
        v_half_2 = verlet_v(dt=dt_R, model=LZ_H, mass=mass,
                            v_curr=v_new,
                            q_curr=q_next_1,
                            active_state=active_state_new)
        q_next_2 = verlet_X(dt=dt_R,
                            q_curr=q_next_1,
                            v_half=v_half_2)
        v_next_2 = verlet_v(dt=dt_R, model=LZ_H,
                            mass=mass,
                            v_curr=v_half_2,
                            q_curr=q_next_2,
                            active_state=active_state_new)
        C_next_2, coeff_next_2 = local_diabatisation(model=LZ_H,
                                                     snapshot=snapshot_1,
                                                     q_next=q_next_2,
                                                     dt=dt_R)
        
        snapshot_2 = Snapshot(positions=q_next_2,
                              velocities=v_next_2,
                              coefficients=coeff_next_2,
                              active_state=active_state_new,
                              gauge=C_next_2,
                              mass=mass)
        traj_rev.append(snapshot_2)
    else:
        snapshot_next = Snapshot(positions=q_next,
                                 velocities=v_next,
                                 coefficients=coeff_next,
                                 active_state=active_state,
                                 gauge=C_next,
                                 mass=mass)
        traj_rev.append(snapshot_next)

print(traj_rev[-1].positions)
print(traj_rev[-1].velocities)
print(traj_rev[-1].active_state)
print(traj_rev[-1].coefficients)
print(traj_rev[-1].gauge)

Hop search finished after 31 iterations
9.999999999996994
0.0019999999999640553
1
[-2.52661494e-09-2.34548662e-09j  9.99999995e-01-3.97317374e-08j]
[[-9.99999875e-01  4.99999813e-04]
 [ 4.99999813e-04  9.99999875e-01]]
